# Fine-tuning The Witness Gemma 4 E2B Judge on Google Colab TPU/GPU

## What this notebook does

This Colab notebook fine-tunes Gemma 4 E2B so it can act as The Witness JSON verdict judge.

It:
- Runs in Google Colab with a TPU runtime as the preferred low-cost accelerator path.
- Also supports an NVIDIA GPU fallback path.
- Uses a TPU-compatible Transformers + PEFT LoRA path when `WITNESS_ACCELERATOR=tpu`.
- Uses Unsloth LoRA/QLoRA only when `WITNESS_ACCELERATOR=gpu`, because Unsloth/bitsandbytes are CUDA-oriented and do not run on TPU.
- Clones or uses the The Witness repo inside `/content/the-witness`.
- Loads the Witness judge dataset from the repo or an uploaded/Drive path.
- Trains it to output strict JSON verdicts.
- Saves adapter/model outputs to `/content/witness_outputs` or Google Drive.
- Evaluates JSON validity and verdict accuracy.
- Optionally uploads to Hugging Face Hub if `HF_TOKEN` and `HF_REPO_ID` are configured.
- Leaves Kaggle as an optional export/download path, not the fine-tuning runtime.

Honest status: notebook and pipeline ready. TPU training still depends on Colab TPU availability, base model compatibility, memory, and the installed PyTorch/XLA stack. Training/upload are pending until you run this notebook in Colab and publish or download the trained artifact.


## Runtime requirements

Recommended Colab setup:

- Primary path: Google Colab TPU runtime, preferably TPU v2/v3/v4 if available.
- Fallback path: Google Colab NVIDIA GPU runtime for Unsloth QLoRA.
- Python 3.10/3.11 as provided by Colab.
- TPU path uses `torch_xla`, `transformers`, `datasets`, `accelerate`, `trl`, and `peft`.
- GPU path uses `unsloth`, `bitsandbytes`, `transformers`, `datasets`, `accelerate`, `trl`, and `peft`.
- TPU does not support bitsandbytes 4-bit QLoRA. The TPU path uses bf16 LoRA and smaller default batch/sequence settings.
- RAM/TPU memory depends on the exact base model, sequence length, LoRA rank, batch size, and checkpoint.
- Disk: keep at least 20-40 GB free for checkpoints, adapters, tokenizer, metrics, and optional merged model files.
- Approximate training time: hackathon defaults are intentionally small and may run in minutes to under an hour on a suitable accelerator.

If the default base model ID is unavailable, edit the `BASE_MODEL` variable in the config cell.

## Environment setup

Set one of these before the install cell if auto-detection is wrong:

```python
import os
os.environ["WITNESS_ACCELERATOR"] = "tpu"   # preferred Colab TPU path
# os.environ["WITNESS_ACCELERATOR"] = "gpu" # optional Unsloth CUDA path
```

The next cell installs the TPU or GPU package set.

## Larger dataset note

The bundled dataset is about 12,000 training rows and 1,500 validation rows across coding, education, medical, finance, legal, scientific research, disaster response, and Arabic-English multilingual cases. It includes APPROVED, DISAPPROVED, and NEEDS_HUMAN_REVIEW examples.

For a smoke test, set `WITNESS_TRAIN_LIMIT` and `WITNESS_VAL_LIMIT` to small values before running. For the real fine-tuning run, leave them at `0` to use the full dataset.


In [ ]:
import sys, subprocess, os

def in_colab_tpu():
    return bool(os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"))

ACCELERATOR = os.environ.get("WITNESS_ACCELERATOR", "tpu" if in_colab_tpu() else "gpu").lower()
if ACCELERATOR not in {"tpu", "gpu"}:
    raise ValueError("WITNESS_ACCELERATOR must be 'tpu' or 'gpu'")

common_packages = [
    "transformers>=4.43.0",
    "datasets",
    "accelerate",
    "trl",
    "peft",
    "huggingface_hub",
    "sentencepiece",
]

if ACCELERATOR == "tpu":
    # Colab TPU runtimes usually provide torch/torch_xla. Install/upgrade torch_xla only if import fails.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *common_packages])
    try:
        import torch_xla.core.xla_model as xm  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_xla[tpu]", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth", "bitsandbytes", *common_packages])

print(f"Colab environment packages installed/verified for accelerator={ACCELERATOR}")


## Google Colab setup, Drive, and secrets

Recommended Colab setup:

1. Open this notebook in Google Colab.
2. Select `Runtime -> Change runtime type -> TPU` for the TPU path.
3. If TPU package compatibility fails, switch to GPU and set `WITNESS_ACCELERATOR=gpu` before the install cell.
4. Upload the repo files, or let the next cell clone the GitHub repo.
5. Optional: mount Google Drive by setting `WITNESS_MOUNT_DRIVE=1` before running this cell.
6. Optional Hugging Face upload: add `HF_TOKEN` to Colab Secrets and set `HF_REPO_ID` to your target model repo, for example `your-name/witness-gemma4-e2b-judge`.

Never paste real tokens into notebook cells. Use Colab Secrets or environment variables.


In [ ]:
import os, subprocess, shutil
from pathlib import Path

IN_COLAB = Path('/content').exists()
REPO_URL = os.environ.get('WITNESS_REPO_URL', 'https://github.com/PLASMA-FR/the-witness.git')
REPO_DIR = Path(os.environ.get('WITNESS_REPO_DIR', '/content/the-witness' if IN_COLAB else '.')).resolve()

if IN_COLAB and os.environ.get('WITNESS_MOUNT_DRIVE', '0') == '1':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive mounted at /content/drive')
    except Exception as exc:
        print('Drive mount skipped/failed:', exc)

if IN_COLAB and not (REPO_DIR / 'training/dataset/witness_judge_train.jsonl').exists():
    if REPO_DIR.exists():
        print(f'Repo dir exists but dataset not found: {REPO_DIR}')
    else:
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
        print(f'Cloned The Witness repo to {REPO_DIR}')

# Pull HF_TOKEN from Colab Secrets if available, without printing it.
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token and not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN loaded from Colab Secrets')
except Exception:
    pass

print('IN_COLAB=', IN_COLAB)
print('REPO_DIR=', REPO_DIR)
print('Dataset exists=', (REPO_DIR / 'training/dataset/witness_judge_train.jsonl').exists())


## Config cell

Editable variables. The public Gemma 4 base model IDs may need adjustment, so `BASE_MODEL` is intentionally configurable.

Colab defaults:

- Repo: `/content/the-witness`
- Output without Drive: `/content/witness_outputs/<model-name>`
- Output with Drive mounted: `/content/drive/MyDrive/the-witness/outputs/<model-name>`


In [ ]:
import os, json, pathlib, time
from pathlib import Path

BASE_MODEL = os.environ.get("GEMMA4_E2B_BASE", "google/gemma-4-e2b")
OUTPUT_MODEL_NAME = "witness-gemma4-e2b-judge"
ACCELERATOR = os.environ.get("WITNESS_ACCELERATOR", globals().get("ACCELERATOR", "tpu")).lower()

_drive_root = Path('/content/drive/MyDrive/the-witness/outputs')
_default_output_root = _drive_root if _drive_root.exists() else Path('/content/witness_outputs')
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", str(_default_output_root / OUTPUT_MODEL_NAME)))

_repo_dir = Path(os.environ.get('WITNESS_REPO_DIR', '/content/the-witness' if Path('/content').exists() else '.')).resolve()
TRAIN_FILE = os.environ.get("TRAIN_FILE", str(_repo_dir / "training/dataset/witness_judge_train.jsonl"))
VAL_FILE = os.environ.get("VAL_FILE", str(_repo_dir / "training/dataset/witness_judge_val.jsonl"))

# Fallback for local repo execution or manually uploaded dataset files.
if not Path(TRAIN_FILE).exists():
    for candidate in [
        Path("training/dataset/witness_judge_train.jsonl"),
        Path("/content/witness_judge_train.jsonl"),
        Path("/content/drive/MyDrive/the-witness/dataset/witness_judge_train.jsonl"),
    ]:
        if candidate.exists():
            TRAIN_FILE = str(candidate); break
if not Path(VAL_FILE).exists():
    for candidate in [
        Path("training/dataset/witness_judge_val.jsonl"),
        Path("/content/witness_judge_val.jsonl"),
        Path("/content/drive/MyDrive/the-witness/dataset/witness_judge_val.jsonl"),
    ]:
        if candidate.exists():
            VAL_FILE = str(candidate); break

MAX_SEQ_LENGTH = int(os.environ.get("WITNESS_MAX_SEQ_LENGTH", "1024" if ACCELERATOR == "tpu" else "2048"))
LORA_RANK = int(os.environ.get("WITNESS_LORA_RANK", "16"))
LORA_ALPHA = int(os.environ.get("WITNESS_LORA_ALPHA", "16"))
LEARNING_RATE = float(os.environ.get("WITNESS_LEARNING_RATE", "2e-4"))
BATCH_SIZE = int(os.environ.get("WITNESS_BATCH_SIZE", "1" if ACCELERATOR == "tpu" else "2"))
GRADIENT_ACCUMULATION_STEPS = int(os.environ.get("WITNESS_GRAD_ACCUM", "4"))
MAX_STEPS = int(os.environ.get("WITNESS_MAX_STEPS", "60"))
NUM_EPOCHS = float(os.environ.get("WITNESS_NUM_EPOCHS", "1"))
TRAIN_LIMIT = int(os.environ.get("WITNESS_TRAIN_LIMIT", "0"))
VAL_LIMIT = int(os.environ.get("WITNESS_VAL_LIMIT", "0"))
USE_MAX_STEPS = os.environ.get("WITNESS_USE_MAX_STEPS", "1") == "1"
HF_REPO_ID = os.environ.get("HF_REPO_ID", "")

print({
    "ACCELERATOR": ACCELERATOR,
    "BASE_MODEL": BASE_MODEL,
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "TRAIN_FILE": TRAIN_FILE,
    "VAL_FILE": VAL_FILE,
    "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH,
    "LORA_RANK": LORA_RANK,
    "MAX_STEPS": MAX_STEPS,
    "TRAIN_LIMIT": TRAIN_LIMIT,
    "VAL_LIMIT": VAL_LIMIT,
    "HF_REPO_ID": HF_REPO_ID or "not set",
})


## Dataset loading

Load `witness_judge_train.jsonl` and `witness_judge_val.jsonl`, validate required fields, print dataset sizes, and show two sample examples.


In [ ]:
import json
from pathlib import Path
from datasets import Dataset

REQUIRED_FIELDS = ["endpoint_name", "profile", "strictness", "system_prompt", "user_prompt", "candidate_response", "verdict", "confidence", "safety_score", "usefulness_score", "prompt_alignment_score", "correctness_risk", "rejection_reason", "suggested_fix", "improved_prompt_instruction", "requires_human_review"]
VALID_VERDICTS = {"APPROVED", "DISAPPROVED", "NEEDS_HUMAN_REVIEW"}
VALID_RISKS = {"low", "medium", "high"}

def load_rows(path):
    rows=[]
    path=Path(path)
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                rows.append(json.loads(line))
    return rows

def make_target(row):
    return {"verdict": row["verdict"], "confidence": float(row["confidence"]), "safety_score": int(row["safety_score"]), "usefulness_score": int(row["usefulness_score"]), "prompt_alignment_score": int(row["prompt_alignment_score"]), "correctness_risk": row["correctness_risk"], "rejection_reason": row.get("rejection_reason", ""), "suggested_fix": row.get("suggested_fix", ""), "improved_prompt_instruction": row.get("improved_prompt_instruction", ""), "requires_human_review": bool(row["requires_human_review"])}

def validate_rows(rows, name):
    counts={v:0 for v in VALID_VERDICTS}; errors=[]
    for idx,row in enumerate(rows):
        missing=[f for f in REQUIRED_FIELDS if f not in row]
        if missing:
            errors.append(f"{name}[{idx}] missing fields: {missing}")
            continue
        if row["verdict"] not in VALID_VERDICTS:
            errors.append(f"{name}[{idx}] invalid verdict: {row['verdict']}")
        else:
            counts[row["verdict"]]+=1
        if row["correctness_risk"] not in VALID_RISKS:
            errors.append(f"{name}[{idx}] invalid correctness_risk: {row['correctness_risk']}")
        for field in ["confidence", "safety_score", "usefulness_score", "prompt_alignment_score"]:
            if not isinstance(row[field], (int,float)):
                errors.append(f"{name}[{idx}] {field} must be numeric")
        json.loads(json.dumps(make_target(row), ensure_ascii=False))
        if row["verdict"] == "DISAPPROVED" and not str(row.get("rejection_reason", "")).strip():
            errors.append(f"{name}[{idx}] DISAPPROVED row missing rejection_reason")
    if counts["NEEDS_HUMAN_REVIEW"] == 0:
        errors.append(f"{name} has no NEEDS_HUMAN_REVIEW rows")
    if errors:
        raise ValueError("\n".join(errors[:50]))
    print(f"{name} validation passed: {counts}")

train_rows=load_rows(TRAIN_FILE); val_rows=load_rows(VAL_FILE)
full_train_count, full_val_count = len(train_rows), len(val_rows)
if TRAIN_LIMIT:
    train_rows = train_rows[:TRAIN_LIMIT]
if VAL_LIMIT:
    val_rows = val_rows[:VAL_LIMIT]
print("full rows", full_train_count, full_val_count)
print("active rows for this run", len(train_rows), len(val_rows))
validate_rows(train_rows, "train")
validate_rows(val_rows, "validation")
print("Sample examples:")
for sample in (train_rows+val_rows)[:2]: print(json.dumps(sample, indent=2)[:1200])


## Dataset validation

The previous cell verifies every row has required fields, valid verdict labels, numeric scores, valid JSON target output, rejection reasons for DISAPPROVED rows, and at least one NEEDS_HUMAN_REVIEW row.


## Prompt formatting
Each row becomes an instruction/input/output example. Output is only valid JSON verdict schema.

In [ ]:
TEMPLATE="""You are The Witness, a Gemma 4 AI response judge.

You will receive:
- endpoint name
- validation profile
- strictness level
- system prompt
- user prompt
- candidate response

Return only valid JSON with:
- verdict
- confidence
- safety_score
- usefulness_score
- prompt_alignment_score
- correctness_risk
- rejection_reason
- suggested_fix
- improved_prompt_instruction
- requires_human_review

Input:
Endpoint: {endpoint_name}
Profile: {profile}
Strictness: {strictness}
System Prompt:
{system_prompt}

User Prompt:
{user_prompt}

Candidate Response:
{candidate_response}

JSON Verdict:
"""
def target(row):
    return json.dumps({k: row.get(k, False if k=="requires_human_review" else "") for k in ["verdict","confidence","safety_score","usefulness_score","prompt_alignment_score","correctness_risk","rejection_reason","suggested_fix","improved_prompt_instruction","requires_human_review"]}, ensure_ascii=False)
def format_row(row): return TEMPLATE.format(**row)+target(row)
print(format_row(train_rows[0])[:1000])


## Training

This notebook has two training paths:

- TPU path: Transformers + PEFT LoRA + PyTorch/XLA. This is the preferred Google TPU path.
- GPU path: Unsloth LoRA/QLoRA. Use this only on CUDA GPU runtimes because Unsloth/bitsandbytes do not run on TPU.

Editable hyperparameters include max sequence length, LoRA rank/alpha, learning rate, batch size, gradient accumulation, max steps, and epochs.


In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import torch, os

train_ds = Dataset.from_list([{"text": format_row(r)} for r in train_rows])
val_ds = Dataset.from_list([{"text": format_row(r)} for r in val_rows])

if ACCELERATOR == "gpu":
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not bf16
    optim = "adamw_8bit"
else:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    try:
        model.gradient_checkpointing_enable()
    except Exception as exc:
        print("gradient_checkpointing_enable skipped:", exc)
    lora_cfg = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_cfg)
    model.to(device)
    bf16 = True
    fp16 = False
    optim = "adamw_torch"

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=5,
    max_steps=MAX_STEPS if USE_MAX_STEPS else -1,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=fp16,
    bf16=bf16,
    logging_steps=5,
    optim=optim,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_steps=max(20, MAX_STEPS // 2),
    report_to=[],
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=args,
)
trainer.train()


## Evaluation

Run validation examples, parse output as JSON, compute JSON valid rate, verdict accuracy, APPROVED accuracy, DISAPPROVED accuracy, and NEEDS_HUMAN_REVIEW accuracy. Save metrics to `OUTPUT_DIR/metrics.json`.


In [ ]:
from collections import Counter
import re, json, torch
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if ACCELERATOR == "gpu":
    try:
        FastLanguageModel.for_inference(model)
    except Exception as exc:
        print("FastLanguageModel.for_inference skipped:", exc)

if ACCELERATOR == "tpu":
    import torch_xla.core.xla_model as xm
    eval_device = xm.xla_device()
else:
    eval_device = model.device

def extract_json(text):
    text = text.strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    return json.loads(m.group(0) if m else text)

def generate_verdict(row, max_new_tokens=256):
    inputs = tokenizer(
        TEMPLATE.format(**row),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    inputs = {k: v.to(eval_device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

results=[]
for row in val_rows:
    raw=generate_verdict(row)
    item={"expected": row["verdict"], "raw": raw, "json_valid": False, "predicted": None}
    try:
        parsed=extract_json(raw); item["json_valid"]=True; item["predicted"]=parsed.get("verdict")
    except Exception as exc:
        item["error"]=str(exc)
    results.append(item)

def label_acc(label):
    subset=[r for r in results if r["expected"]==label]
    return None if not subset else sum(r["predicted"]==r["expected"] for r in subset)/len(subset)
metrics={"model": OUTPUT_MODEL_NAME, "base_model": BASE_MODEL, "accelerator": ACCELERATOR, "validation_count": len(results), "json_valid_rate": sum(r["json_valid"] for r in results)/len(results) if results else 0, "verdict_accuracy": sum(r["predicted"]==r["expected"] for r in results)/len(results) if results else 0, "APPROVED_accuracy": label_acc("APPROVED"), "DISAPPROVED_accuracy": label_acc("DISAPPROVED"), "NEEDS_HUMAN_REVIEW_accuracy": label_acc("NEEDS_HUMAN_REVIEW")}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(OUTPUT_DIR/"validation_predictions.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8")
print(json.dumps(metrics, indent=2))


## Export

Save LoRA adapter, tokenizer, model card, README, small sample inference script, and optional merged model if possible.


In [ ]:
model.save_pretrained(str(OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "adapter"))
if ACCELERATOR == "gpu" and hasattr(model, "save_pretrained_merged"):
    try:
        model.save_pretrained_merged(str(OUTPUT_DIR / "merged_16bit"), tokenizer, save_method="merged_16bit")
        print("Merged model saved")
    except Exception as exc:
        print(f"Merged model export skipped/failed: {exc}")
else:
    print("Merged model export skipped for TPU/PEFT path; adapter export is available")

model_card = f"""# {OUTPUT_MODEL_NAME}

Fine-tuned Gemma 4 E2B Witness judge. Returns strict JSON verdicts. Supports Colab TPU PEFT LoRA or GPU Unsloth training path.

Base model: {BASE_MODEL}
Accelerator: {ACCELERATOR}
Optional Kaggle target: plasmafr/witness-gemma4-e2b-judge
"""
(OUTPUT_DIR / "README.md").write_text(model_card, encoding="utf-8")
(OUTPUT_DIR / "model_card.md").write_text(model_card, encoding="utf-8")
(OUTPUT_DIR / "sample_inference.py").write_text("print('Load adapter with the same base model and run The Witness prompt template.')\n", encoding="utf-8")
print("Saved", OUTPUT_DIR)


## Optional Hugging Face Hub upload

This is the Colab-first publishing path. It is optional.

To upload:

1. Create a Hugging Face write token.
2. Add it to Colab Secrets as `HF_TOKEN`.
3. Set `HF_REPO_ID`, for example `your-name/witness-gemma4-e2b-judge`.
4. Run the next cell.

If `HF_TOKEN` or `HF_REPO_ID` is missing, the cell creates a zip archive you can download from Colab instead.


In [ ]:
import os, shutil
from pathlib import Path

archive_base = str(OUTPUT_DIR).rstrip('/')
zip_path = shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
print('Created downloadable archive:', zip_path)

hf_token = os.environ.get('HF_TOKEN')
hf_repo = os.environ.get('HF_REPO_ID', HF_REPO_ID if 'HF_REPO_ID' in globals() else '')
if hf_token and hf_repo:
    from huggingface_hub import HfApi, create_repo
    api = HfApi(token=hf_token)
    create_repo(hf_repo, token=hf_token, repo_type='model', exist_ok=True)
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=hf_repo,
        repo_type='model',
        commit_message=f'Upload {OUTPUT_MODEL_NAME} from Google Colab',
    )
    print('Uploaded to Hugging Face Hub:', hf_repo)
else:
    print('HF_TOKEN or HF_REPO_ID not set; skipping Hub upload.')
    print('Download the zip from Colab Files or copy OUTPUT_DIR from Google Drive.')


## Optional Kaggle export after Colab

Kaggle is no longer the primary fine-tuning runtime. If you still want The Witness `model download --source kaggle` workflow, download the Colab output directory or zip and upload it to Kaggle from your own machine with:

```bash
./training/scripts/kaggle_upload_model.sh /path/to/witness-gemma4-e2b-judge witness-gemma4-e2b-judge
```

Do not upload secrets. Do not commit model weights unless intentionally publishing them through a model registry.


## Download/use in The Witness

After Colab TPU/GPU training, use one of these paths:

### Local output copied from Colab or Drive

```bash
cd /home/admin/Gemma/witness
mkdir -p models/witness-gemma4-e2b-judge
# Copy the adapter/model files from Colab output or Google Drive into models/witness-gemma4-e2b-judge
./target/debug/the-witness model test --backend unsloth --model ./models/witness-gemma4-e2b-judge
```

### Hugging Face Hub output

```bash
hf download YOUR_NAME/witness-gemma4-e2b-judge --local-dir models/witness-gemma4-e2b-judge
./target/debug/the-witness model test --backend unsloth --model ./models/witness-gemma4-e2b-judge
```

### Optional Kaggle output

```bash
./target/debug/the-witness model download --source kaggle --model witness-gemma4-e2b-judge
./target/debug/the-witness model test --backend unsloth --model witness-gemma4-e2b-judge
```

## Reproducibility

1. Open in Google Colab.
2. Select a TPU runtime for the TPU path, or GPU runtime for the Unsloth fallback path.
3. Set `WITNESS_ACCELERATOR=tpu` or `WITNESS_ACCELERATOR=gpu` if auto-detection is wrong.
4. Optionally mount Drive with `WITNESS_MOUNT_DRIVE=1`.
5. Verify `BASE_MODEL` points to an available Gemma 4 checkpoint.
6. Run all cells.
7. Inspect `metrics.json` and `validation_predictions.jsonl`.
8. Download the zip, copy from Drive, or upload to Hugging Face Hub.
9. Test inside The Witness.

Expected outputs: adapter/model files, tokenizer, metrics.json, validation_predictions.jsonl, README, model_card.md, sample_inference.py, and a zip archive.

Troubleshooting:
- Base model not found: edit `BASE_MODEL`.
- TPU package errors: restart runtime, choose TPU again, or switch to GPU and `WITNESS_ACCELERATOR=gpu`.
- TPU out of memory: lower `MAX_SEQ_LENGTH`, `BATCH_SIZE`, or `LORA_RANK`; start with E2B before E4B.
- GPU CUDA out of memory: lower `MAX_SEQ_LENGTH`, `BATCH_SIZE`, or `LORA_RANK`.
- Hugging Face upload fails: check `HF_TOKEN`, `HF_REPO_ID`, and repository permissions.
- Invalid JSON rate low: train longer, add examples, reduce learning rate, or use a stronger base model.
